This notebook reads the finished five-fold mean for patient1. It does not launch a GPU job and does not re-run the single-checkpoint pipeline. Restart Kernel & Run All.

Plotting and table code is in `patient1_ens5.py`. The cells below only call it. Full-resolution PDF and PNG files are written under `figures/`. Each cell shows a reduced preview, so this notebook does not store the full raster.


# NPC external validation — cohort_safe `l12_links_1` seed 42, five-fold mean

This notebook shows **patient1** only (`20210809-C-AH4199551`). The figure layout matches `Pred_visual_npc_uni2_A2.ipynb`: L1 / L12 / L2 composition and spatial maps, B-cell confidence, Visium spot versus cell2location, the expert TLS zoom, and TLS / SRI / TNI.

The labels are the mean softmax of the five fold weights `hce_group_fold_0.pt` … `hce_group_fold_4.pt` from `result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42` (`INNER_VAL=cohort_safe`, training seed 42). This is not the older A2 `best_mlp`, and not the single fold-4 file in `pred_uni2_axisfix_l12_links_1_ivsafe_seed42`.

`STAR_PROB_MIN=0`, so the folder has no `_prob` suffix. UNI2 tokens stay in `project_all_UNI2`. There is no nucleus-level cell-type ground truth and no ROC.

- predictions: `data/Visium/NPC/pred_uni2_ens5_l12_links_1_ivsafe_seed42/{sample}/`
- training OOF used for the B-cell FPR gate: `data/PanCancer/result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42/oof_predictions.npz`


## Predictions already written

```bash
RUNS="data/PanCancer/result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42" \
  CUDA_DEVICE=2 bash code/External_val/predict_fold_ensemble.sh NPC
```

That command writes `pred_uni2_ens5_l12_links_1_ivsafe_seed42/`. This notebook only reads it. With one `RUNS` entry the script does not also write the ten-model mean.

Confidence sidecar (CPU, no GPU; patient1). The OOF file is this seed's training run, the non-B null for the FPR gate:

```bash
PY=/nobackup2/users/lingyu/conda_envs/SeededNTM/bin/python
PRED=data/Visium/NPC/pred_uni2_ens5_l12_links_1_ivsafe_seed42
OOF=data/PanCancer/result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42/oof_predictions.npz
SAMPLE=20210809-C-AH4199551
"${PY}" code/Hist2Pheno_pkg/prediction_confidence.py \
  "${PRED}/${SAMPLE}/${SAMPLE}_l2_probs.npz" --oof "${OOF}" \
  --classes B_cell Plasma B_Plasma --fdr 0.10
```


In [ ]:
import sys
from pathlib import Path

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path[:0] = [str(REPO / "code"), str(REPO / "code" / "External_val" / "NPC")]

from patient1_ens5 import open_patient1

run = open_patient1(REPO)


In [ ]:
run.composition()


In [ ]:
run.spatial_maps()


## Each fold's L2 map

The five-fold mean in the panel above is the average of these five L2 softmaxes. Each panel is one weight, `hce_group_fold_0.pt` through `hce_group_fold_4.pt`. The color is the argmax of that fold's own L2 head probabilities (`{sample}_l2_probs.npz`), not the averaged probabilities, the decision rule, or prior shift.

Fold 4 is the checkpoint selected on this run's inner validation. That comparison is among the training cohorts. This external slide was not one of the five folds, so the validation winner is not a model chosen for patient1.


In [ ]:
run.fold_l2_maps()


## B-cell presence vs argmax (UTOPIA-style confidence)

Argmax `pred_CT` hides B / Plasma when Tumor or DC wins. Sidecar `{sample}_confidence.npz` is written by `prediction_confidence.py` from the existing L2 softmax and pan-cancer OOF (no GT on these slides). **Presence** is an FPR gate vs OOF non-B cells (`--fdr 0.10` = null 90th percentile), not whole-slide BH.

If the sidecar is missing:

```bash
PY=/nobackup2/users/lingyu/conda_envs/SeededNTM/bin/python
PRED=data/Visium/NPC/pred_uni2_ens5_l12_links_1_ivsafe_seed42
OOF=data/PanCancer/result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42/oof_predictions.npz
SAMPLE=20210809-C-AH4199551
"${PY}" code/Hist2Pheno_pkg/prediction_confidence.py \
  "${PRED}/${SAMPLE}/${SAMPLE}_l2_probs.npz" --oof "${OOF}" \
  --classes B_cell Plasma B_Plasma --fdr 0.10
```

Columns: argmax B/Plasma · FPR≤0.10 presence · conformal confidence · **present × confidence** (same score, FPR gate applied; not a second independent test).


In [ ]:
run.bcell_presence()


## Resolution from Visium spot to single cell

Same mapping as `Fig2_Fig3_panels.ipynb` (Fig. 3b). Each StarDist nucleus goes to the **nearest** Visium spot and is kept only inside that spot’s capture disc (`spot_diameter_fullres / 2`), via `Hist2Pheno_pkg/nuclei_to_spots.py`.

Three spatial panels: **single-nucleus L2**, **pseudo-spot** (majority predicted L2 in the disc), **cell2location** (argmax of the 7 abundances). Patient1 uses `Annotation/sp.obs.csv` (`AH4199551`); patient2 uses `Annotation/AJ_sp.obs.csv` (`AJ3037946`). The next cell stacks Hist2Pheno vs pseudo-spot vs cell2location on the same 7-type vocabulary.


In [ ]:
run.visium_resolution()


In [ ]:
# Per-type abundance, not argmax. Each panel uses its own 99th percentile.
run.abundance_vs_cell2location()


In [ ]:
# Left off. One row per L2 type: softmax and cluster conformal confidence.
# run.nucleus_confidence()


### Stacked composition (same 7 types)

All three bars are **class proportions** on the same 7 cell2location types. Hist2Pheno = fraction of nuclei after mapping L2 → these types. Pseudo-spot = fraction of spots by majority mapped L2. cell2location = fraction of spots after converting per-spot abundances to a class (`argmax`). `other` is Endothelial / Neural (Hist2Pheno only).


In [ ]:
run.stacked_composition()


## TLS annotation (patient1)

Expert Visium TLS barcodes in `data/Visium/NPC/Annotation/TLS_spot_label.csv` (36 spots on `AH4199551` / `20210809-C-AH4199551`). Each barcode is mapped through `patient1/tissue_positions_list.csv`. A Gaussian-smoothed capture disc around those spots gives a **smooth outline** of each TLS focus. **Only the three left-side foci are kept** (the two lower-right clusters are dropped). The zoom box is the bounding rectangle of the kept spots plus padding. Overview: black outline only. Zoom: black outline + Visium spots. Then Hist2Pheno L1 / L12 / L2 inside that zoom. The L1 / L12 / L2 legend lists only nuclei inside the black TLS outlines, not every nucleus in the zoom rectangle. A 100% stacked bar uses that same outline set.


In [ ]:
run.tls_annotation()


## TLS zoom, L2 softmax abundance

Same crop and the same three expert outlines as the panels above. Each panel sums L2 softmax columns of the five-fold mean in `{sample}_l2_probs.npz`. This is the same quantity as `P(B_cell)+P(Plasma)` in `Pred_visual_npc_patient1_axisfix.ipynb`: the stored softmax, not argmax, not `decision_rule_l2_only.json`, and not prior shift.

| Panel | Columns |
|--|--|
| B cell | B_cell + Plasma |
| T cell | CD4_T + CD8_T + Tregs |
| DC | DC |
| Tumor | Tumor |

NK is left out of T cell, and DCIS is left out of Tumor. Each panel uses its own 99th percentile on this slide, because the four sums sit on different ranges. The colorbar is that probability. The title is the mean inside the black outline versus the mean outside it. The following cell keeps that color inside the outline and draws the other nuclei in the crop light gray.


In [ ]:
run.tls_softmax_abundance()


In [ ]:
run.tls_softmax_abundance(gray_outside=True)


## Histology-derived niche indices (TLS / SRI / TNI)

Same implementation as `code/Xenium_lung/histology_derived_niche_index.ipynb` §3, via `Hist2Pheno_pkg/histology_niche_index.py`. Lung ARI / FRI are not used: those members (AT1/AT2, activated fibrotic FBs, SPP1+ macrophages) are not pan-cancer L2 classes.

Scores use **L2 softmax**, not the argmax label. The softmax is the five-fold mean already stored in `{sample}_l2_probs.npz`.

| Index | Abundance | Spatial (radius R) |
|--|--|--|
| **TLS** | sum(B_cell + Plasma + CD4_T + CD8_T + Tregs + DC) | `(B_local × T_local × DC_local)^(1/3)` |
| **SRI** | (Fibroblast + Myofibroblast + Stromal) / epithelium | Fib_local / Epi_local |
| **TNI** | (fibroblast + Macrophage + Endothelial) / epithelium | `(Fib × Mac × Endo)^(1/3) / Epi_local` |

Epithelium = Epithelial, Tumor, Alveolar, Injury_epithelial, DCIS, Myoepithelial. NK is left out of TLS, as in the lung T-cell set. `spatial_HE` is native HE pixels, so R = **55 µm** is converted with the OME pixel size **0.465 µm/px** (`55 / 0.465 ≈ 118 px`), not 0.5. Hotspots are the within-slide 95th percentile. TLS, SRI, and TNI each use the same 1×3 row: abundance, spatial score without markers, then the same spatial map with cyan candidate nuclei. The two spatial panels share one color scale.


In [ ]:
run.niche_formulas()


In [ ]:
run.niche_indices()


In [ ]:
run.niche_maps()
